In [4]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import normalize
from sklearn.model_selection import train_test_split

# Paths
image_dir = r"C:\project_mrsac\DATASET\train\images"  # Folder with original satellite images
mask_dir = r"C:\project_mrsac\DATASET\mask_img"    # Folder with binary mask images

# Load images & masks
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png') or f.endswith('.jpg')])
mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])

# Resize images/masks to match input shape (256x256)
IMG_SIZE = 256
X, Y = [], []

for img_name, mask_name in zip(image_files, mask_files):
    img = cv2.imread(os.path.join(image_dir, img_name))
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = normalize(img, axis=1)  # Normalize pixel values (0-1)
    
    mask = cv2.imread(os.path.join(mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = mask / 255.0  # Convert to binary (0 or 1)

    X.append(img)
    Y.append(mask)

# Convert lists to numpy arrays
X = np.array(X)
Y = np.array(Y).reshape(-1, IMG_SIZE, IMG_SIZE, 1)  # Add channel dimension

# Split dataset (80% train, 20% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"✅ Dataset prepared: {len(X_train)} train images, {len(X_test)} test images")


✅ Dataset prepared: 72 train images, 18 test images


In [5]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate

def build_unet(input_size=(256, 256, 3)):
    inputs = Input(input_size)
    
    # Encoder
    c1 = Conv2D(64, (3, 3), activation="relu", padding="same")(inputs)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(128, (3, 3), activation="relu", padding="same")(p1)
    p2 = MaxPooling2D((2, 2))(c2)
    
    c3 = Conv2D(256, (3, 3), activation="relu", padding="same")(p2)
    p3 = MaxPooling2D((2, 2))(c3)
    
    c4 = Conv2D(512, (3, 3), activation="relu", padding="same")(p3)
    
    # Decoder
    u1 = UpSampling2D((2, 2))(c4)
    m1 = concatenate([u1, c3])
    c5 = Conv2D(256, (3, 3), activation="relu", padding="same")(m1)
    
    u2 = UpSampling2D((2, 2))(c5)
    m2 = concatenate([u2, c2])
    c6 = Conv2D(128, (3, 3), activation="relu", padding="same")(m2)
    
    u3 = UpSampling2D((2, 2))(c6)
    m3 = concatenate([u3, c1])
    c7 = Conv2D(64, (3, 3), activation="relu", padding="same")(m3)

    outputs = Conv2D(1, (1, 1), activation="sigmoid")(c7)  # Binary segmentation

    return Model(inputs, outputs)

# Create U-Net model
unet = build_unet()
unet.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
unet.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 32,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (UpSampling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, 64,    │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 768)              │            │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │  1,769,728 │ concatenate[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 128, 128,  │          0 │ conv2d_4[0][0]    │
│ (UpSampling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128, 128,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 384)              │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 128, 128,  │    442,496 │ concatenate_1[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_2     │ (None, 256, 256,  │          0 │ conv2d_5[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 256, 256,  │          0 │ up_sampling2d_2[… │
│ (Concatenate)       │ 192)              │            │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 256, 256,  │    110,656 │ concatenate_2[0]

 Total params: 3,873,921 (14.78 MB)

 Trainable params: 3,873,921 (14.78 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the model
history = unet.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=30, batch_size=8)

# Save the trained model
unet.save("farm_boundary_unet.h5")
print("✅ Model training completed & saved!")


Epoch 1/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 63s 6s/step - accuracy: 0.9000 - loss: 0.5469 - val_accuracy: 0.9154 - val_loss: 0.3284
Epoch 2/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 57s 6s/step - accuracy: 0.9115 - loss: 0.3027 - val_accuracy: 0.9154 - val_loss: 0.2659
Epoch 3/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 87s 7s/step - accuracy: 0.9104 - loss: 0.2621 - val_accuracy: 0.9154 - val_loss: 0.2393
Epoch 4/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 59s 7s/step - accuracy: 0.9100 - loss: 0.2508 - val_accuracy: 0.9154 - val_loss: 0.2300
Epoch 5/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 58s 6s/step - accuracy: 0.9086 - loss: 0.2471 - val_accuracy: 0.9154 - val_loss: 0.2280
Epoch 6/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 57s 6s/step - accuracy: 0.9112 - loss: 0.2396 - val_accuracy: 0.9154 - val_loss: 0.2286
Epoch 7/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 59s 7s/step - accuracy: 0.9102 - loss: 0.2412 - val_accuracy: 0.9154 - val_loss: 0.2263
Epoch 8/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 57s 6s/step - accuracy: 0.9126 - loss: 0.2364 - val_accuracy: 0.9154 - val_loss: 0.2256
Epoch 9/

✅ Model training completed & saved!


In [1]:
import matplotlib.pyplot as plt

# Load test image
test_img = cv2.imread(os.path.join(image_dir, image_files[1]))  # Pick a test image
test_img = cv2.resize(test_img, (256, 256))
test_img = normalize(test_img, axis=1)
test_img = np.expand_dims(test_img, axis=0)  # Add batch dimension

# Predict mask
predicted_mask = unet.predict(test_img)[0]
predicted_mask = (predicted_mask > 0.5).astype(np.uint8)  # Convert probabilities to binary mask

# Show original & predicted mask
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.imread(os.path.join(image_dir, image_files[0])))  # Original image
plt.title("Original Image")
plt.subplot(1, 2, 2)
plt.imshow(predicted_mask, cmap="gray")  # Predicted mask
plt.title("Predicted Mask (Farm Boundaries)")
plt.show()


NameError: name 'cv2' is not defined